In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
    
import numpy as np
import scipy
from scipy.sparse import coo_matrix, csr_matrix, csc_matrix, block_diag, identity, hstack
import matplotlib.pyplot as plt
from pyiga import assemble, bspline, vform, geometry,vis, solvers, utils, topology, algebra, quadrature, operators
#from sksparse.cholmod import cholesky
from pyiga import adaptive as adap
import itertools
import time
import statistics as st
from pyiga import algebra_cy, assemble_cy, bspline_cy

np.set_printoptions(linewidth=100000)
np.set_printoptions(precision=5)
np.set_printoptions(formatter={'float_kind':"{:.3f}".format})

In [2]:
def pyx_find_pivot(C, active, num_active, take_max=False):
    pivot = {}

    indptr = C.indptr
    indices = C.indices
    data = C.data

    for i in range(num_active):
        r = active[i]
        elim_dof = -1
        elim_val = 0.0
        feasible = True

        # First pass: choose pivot candidate
        for ind in range(indptr[r], indptr[r + 1]):
            c = indices[ind]
            v = data[ind]

            if take_max:
                if abs(abs(v) - elim_val) < 1e-14:
                    if v > 1e-14:
                        elim_dof = c
                        elim_val = abs(v)

                if abs(v) > elim_val + 1e-14:
                    elim_dof = c
                    elim_val = abs(v)

            else:
                if abs(v - 1.0) < 1e-14:
                    elim_dof = c
                    elim_val = abs(v)

        # Second pass: feasibility check
        for ind in range(indptr[r], indptr[r + 1]):
            c = indices[ind]
            v = data[ind]
            if abs(v) > 1e-14 and c in pivot:
                feasible = False

        if elim_dof == -1:  # empty row
            feasible = False

        if feasible:
            pivot[elim_dof] = r

    return pivot

In [114]:
C = csr_matrix(np.array([[1,-1,0,0,0],[0,1,-1,0,0],[0,0,1,-1,0],[0,0,0,1,-1],[-1,0,0,0,1]]),dtype='float')
C = C[np.random.permutation(C.shape[0])]

In [115]:
Phi,_ = algebra_cy.pyx_compute_basis(*C.shape, C, 10, 0)

In [116]:
Phi.toarray()

array([[1.000],
       [1.000],
       [1.000],
       [1.000],
       [1.000]])

In [92]:
active=pyx_find_pivot(C,np.arange(5),5)

In [93]:
active

{np.int32(4): np.int64(0), np.int32(1): np.int64(1), np.int32(2): np.int64(2)}

In [79]:
Phi = scipy.sparse.identity(5, format="csr")
I = scipy.sparse.identity(5, format="csr")
en = np.array([0,1,0,0,0])[None].T
em = np.array([1,0,0,0,0])[None].T

In [80]:
en

array([[0],
       [1],
       [0],
       [0],
       [0]])

In [ ]:
I - np.sum([1/()])

In [85]:
R = np.array(I-1/(em.T@C@Phi@en)*en@em.T@C@Phi)

In [86]:
R

array([[1.000, 0.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 1.000, 0.000, 0.000],
       [0.000, 0.000, 1.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 1.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 1.000]])

In [87]:
C@R

array([[0.000, 0.000, 0.000, 0.000, 0.000],
       [-0.500, 0.000, 0.000, 0.000, 0.500],
       [0.000, 0.000, 0.000, 0.500, -0.500],
       [0.500, 0.000, -0.500, 0.000, 0.000],
       [0.000, 0.000, 0.500, -0.500, 0.000]])

In [42]:
e1@e1.T@C@Phi

array([[1.000, -1.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000]])

In [43]:
e1@e1.T

array([[1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]])

In [40]:
1/(e1.T@C@Phi@e1)*e1@e1.T@C@Phi

array([[1.000, -1.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000],
       [0.000, 0.000, 0.000, 0.000, 0.000]])

In [21]:
e1@e1.T@C

ValueError: Scalar operands are not allowed, use '*' instead

In [24]:
e1

array([1, 0, 0, 0, 0])